# K3 — Rakip (baseline) koşumu · DermaMNIST · MedMNIST resmî ResNet-18 (28 px, sıfırdan)

Konu 3.1 · hedef D (Colab Pro, **NVIDIA L4** sabit donanım sınıfı) · kuyruk `k3-baseline` (8 koşum, kademeli).

| aşama | varyant | seed | epoch | amaç |
|---|---|---|---|---|
| 1 duman | `baseline_duman` | 0 | 3 | kod/veri/pin yolu çalışıyor mu |
| 2 ön-uçuş | `baseline_onucus` | 0, 1 | 20 | seed bağlı mı, öğreniyor mu, süre makul mü |
| 3 tam | `baseline` | 0–4 | 100 | resmî protokol, replikasyon kapısı için |

Aşama kapıları `kuyruk.py kos --kademeli` içinde **otomatik** koşar (`kuyruk/k3-baseline.beklenti.json`, ön-kayıtlı).
Kapı geçilmezse tam koşum BAŞLAMAZ. O durumda sonucu zipleyip şefe getir.

**Tahmini süre:** yaklaşık 1,5–2 saat (aşama 3 ≈ 15–20 dk/seed). Colab Pro'da arka planda yürütme yok. **Sekmeyi açık tut.**
Oturum koparsa defteri baştan **Tümünü çalıştır** ile koştur. Hücre **5b** bitmemiş kuyruğu kendisi algılar.
5b, bitmiş koşumları atlayıp yalnız kalanları koşar.

Hücreleri **sırayla** çalıştır (Çalışma zamanı → Tümünü çalıştır).

In [ ]:
# 1) Drive'ı bağla, paketi çalışma köküne aç (runs/ ve data/medmnist/ pakette yok → önceki sonuçlar korunur)
from google.colab import drive
drive.mount('/content/drive')
import os, json, zipfile, hashlib, pathlib, subprocess, shutil, time
YZ = pathlib.Path('<DRIVE_KOK>')
KOK = YZ / '3-1-dermamnist-dinov2'
PAKET = YZ / '3-1-paket.zip'
assert PAKET.exists() or (KOK / 'scripts' / 'rakip_kos.py').exists(), \
    f'{PAKET} bulunamadı. OKU-BENI.md adım 2: zip dosyasını Drive > Bilgisayarım > yz klasörüne yükle.'
if PAKET.exists():
    with zipfile.ZipFile(PAKET) as z:
        yeni = json.loads(z.read('PAKET.json'))
        eski_yol = KOK / 'PAKET.json'
        eski = json.loads(eski_yol.read_text(encoding='utf-8')) if eski_yol.exists() else {}
        if eski.get('surum') != yeni['surum']:
            KOK.mkdir(parents=True, exist_ok=True)
            z.extractall(KOK)
            print('paket açıldı → sürüm', yeni['surum'], '·', len(yeni['dosyalar']), 'dosya')
        else:
            print('paket zaten açık → sürüm', yeni['surum'])
os.chdir(KOK)
print('çalışma kökü:', os.getcwd())

def kos(cmd):
    """Komutu koşar, çıktıyı canlı basar, dönüş kodunu verir."""
    print('$', ' '.join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for satir in p.stdout:
        print(satir, end='', flush=True)
    return p.wait()

In [ ]:
# 2) Donanım: yalnız NVIDIA L4 kabul edilir (konu.md — karşılaştırılan TÜM koşumlar aynı sınıfta)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print('GPU:', gpu)
assert 'L4' in gpu, (f'GPU L4 DEĞİL: {gpu!r}. Çalışma zamanı > Çalışma zamanı türünü değiştir > L4 GPU seç, '
                     'sonra defteri baştan çalıştır. Başka GPU ile ölçüm YAPILMAZ.')
mesgul = subprocess.run(['nvidia-smi', '--query-compute-apps=pid,name', '--format=csv,noheader'],
                        capture_output=True, text=True).stdout.strip()
assert not mesgul, f'GPU başka bir süreçle meşgul: {mesgul} (aynı GPU\'da ikinci eğitim yok)'

In [ ]:
# 3) Pinli kurulum (env/requirements.lock ile aynı çekirdek sürümler). Kernel torch'u hiç import etmez → yeniden başlatma gerekmez.
kos(['python', '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu121'])
kos(['python', '-m', 'pip', 'install', '-q', 'medmnist==3.0.2', 'timm==1.0.30', 'scikit-learn==1.9.1', 'numpy==2.2.6', 'tensorboardX==2.6.5'])
dogrula = r"""
import json, platform, torch, torchvision, medmnist, timm, sklearn, numpy, tensorboardX
print(json.dumps({'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
  'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(), 'cuda_var': torch.cuda.is_available(),
  'cihaz': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'medmnist': medmnist.__version__,
  'timm': timm.__version__, 'sklearn': sklearn.__version__, 'numpy': numpy.__version__, 'tensorboardX': tensorboardX.__version__}))
"""
r = subprocess.run(['python', '-c', dogrula], capture_output=True, text=True)
print(r.stderr[-2000:])
ortam = json.loads(r.stdout.strip().splitlines()[-1])
print(json.dumps(ortam, indent=1, ensure_ascii=False))
beklenen = {'torch': '2.5.1', 'torchvision': '0.20.1', 'medmnist': '3.0.2', 'timm': '1.0.30', 'sklearn': '1.9.1',
            'numpy': '2.2.6', 'tensorboardX': '2.6.5'}
for k, v in beklenen.items():
    assert str(ortam[k]).startswith(v), f'{k} pinli sürüm değil: {ortam[k]} ≠ {v} (kurulum hücresini tekrar çalıştır)'
assert ortam['cuda_var'] and 'L4' in ortam['cihaz'], f'torch CUDA/L4 görmüyor: {ortam}'
pathlib.Path('env').mkdir(exist_ok=True)
ortam['nvidia_smi'] = gpu
ortam['zaman'] = time.strftime('%Y-%m-%dT%H:%M:%S')
pathlib.Path('env/ortam-colab.json').write_text(json.dumps(ortam, indent=1, ensure_ascii=False), encoding='utf-8')
pathlib.Path('env/requirements-colab.lock').write_text(
    subprocess.run(['python', '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout, encoding='utf-8')
print('kurulum doğrulandı → env/ortam-colab.json, env/requirements-colab.lock')

In [ ]:
# 4) Veri: DermaMNIST 28 px → data/medmnist (Drive'da kalıcı). MD5 resmî değerle, SHA256 yerel manifestle doğrulanır.
indir = r"""
import os, medmnist
os.makedirs('data/medmnist', exist_ok=True)
medmnist.DermaMNIST(split='train', size=28, root='data/medmnist', download=True)
print('MD5 (medmnist.INFO):', medmnist.INFO['dermamnist']['MD5'])
"""
assert kos(['python', '-c', indir]) == 0, 'indirme başarısız'
manifest = json.loads(pathlib.Path('data/data_manifest.json').read_text(encoding='utf-8'))['dosyalar']['dermamnist.npz']
npz = pathlib.Path('data/medmnist/dermamnist.npz')
h5, h256 = hashlib.md5(), hashlib.sha256()
with open(npz, 'rb') as f:
    for parca in iter(lambda: f.read(1 << 20), b''):
        h5.update(parca); h256.update(parca)
print('md5   ', h5.hexdigest(), '\nsha256', h256.hexdigest(), '\nbayt  ', npz.stat().st_size)
assert h5.hexdigest() == manifest['md5_resmi'] == '0744692d530f8e62ec473284d019b0c7', 'MD5 resmî değerle UYUŞMUYOR — dur'
assert h256.hexdigest() == manifest['sha256'], 'SHA256 yerel manifestle uyuşmuyor — dur'
pathlib.Path('data/indirme_colab.json').write_text(json.dumps({'dosya': str(npz), 'md5': h5.hexdigest(),
    'sha256': h256.hexdigest(), 'bayt': npz.stat().st_size, 'zaman': time.strftime('%Y-%m-%dT%H:%M:%S')}, indent=1), encoding='utf-8')
print('veri doğrulandı')

In [ ]:
# 5) KUYRUK — kademeli (duman → ön-uçuş → tam). Aşama kapıları otomatik; ihlalde tam koşum başlamaz (rc=3).
AD = 'k3-baseline'
done = KOK / 'runs' / 'kuyruklar' / f'{AD}.KUYRUK.done'
baslat = KOK / 'runs' / 'kuyruklar' / f'{AD}.baslat.json'
if done.exists():
    print('Bu kuyruk zaten bitmiş/durmuş — tekrar koşulmaz. Damga:\n', done.read_text(encoding='utf-8')[:1500])
    print('→ hücre 6 ile sonuçları zipla.')
elif baslat.exists():
    print('UYARI: kuyruk önceki bir oturumda BAŞLAMIŞ ama bitmemiş (oturum koptu).')
    print('kuyruk.py bitmiş satırları ATLAMAZ; aynı adı yeniden koşmak tamamlanmış seedleri tekrarlar.')
    print('→ Bu hücreyi geç, 5b (DEVAM) hücresini çalıştır.')
else:
    t0 = time.time()
    rc = kos(['python', 'orkestra/kuyruk.py', '--kok', '.', 'kos', AD, '--kademeli', '--paralel', '1'])
    print(f'\nkuyruk rc={rc} · süre {(time.time() - t0) / 60:.1f} dk  (0=hepsi başarılı, 1=başarısız koşum var, 3=aşama kapısında DURDU)')
    if rc == 3:
        print('Aşama kapısı İHLAL → sonuclar/beklenti-k3-baseline-asama*.md. Tam koşum başlatılmadı. Hücre 6 ile zipla, şefe getir.')

### 5b) DEVAM — oturum koptuysa kendiliğinden devreye girer (normal akışta hiçbir şey yapmaz)
Son kuyruk başlamış ama damgasızsa (oturum koptu), önce ölü koşumlar `rc=96` ile damgalanır (yetim).
Sonra yeni bir kuyruk adıyla (`k3-baseline-devamN`) **yalnız kalan satırlar** koşar.
`kuyruk.py` bitmiş satırları kendisi atlamaz, aynı kuyruk adı da yeniden koşulmaz. Bu yüzden bu hücre var.
- Aşama 2 kapısı daha önce GEÇTİ ise: yalnız başarıyla bitmemiş aşama-3 seed'leri koşar.
- Geçmediyse ya da hiç koşmadıysa: bütün satırlar baştan koşar (aşama 1–2 kısa; kapılar yeni kuyrukta yeniden çalışır).
- Bir kuyruk aşama kapısında DURDU ise devam YOK. Sonucu zipleyip şefe getir.

Beklenti dosyası **bayt bayt aynı** kopyalanır (ön-kayıt değişmez).

In [ ]:
# 5b) DEVAM — yalnız son kuyruk başlamış ama bitmemişse çalışır
AD = 'k3-baseline'
KL = KOK / 'runs' / 'kuyruklar'

def damga(ad):
    p = KL / f'{ad}.KUYRUK.done'
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else None

devamlar = sorted((p.stem for p in (KOK / 'kuyruk').glob(f'{AD}-devam*.jsonl')), key=lambda a: int(a.rsplit('devam', 1)[1]))
kuyruklar = [AD] + devamlar
son = kuyruklar[-1]
durdu = [k for k in kuyruklar if damga(k) and damga(k).get('durduruldu_asama') is not None]
baslamis = (KL / f'{son}.baslat.json').exists()
if damga(son) is not None or (son == AD and not baslamis):
    print(f'DEVAM GEREKMİYOR: son kuyruk {son} ' + ('bitmiş.' if damga(son) else 'hiç başlamamış (hücre 5 koşar).') + ' → hücre 6')
elif durdu:
    print(f'Kuyruk(lar) aşama kapısında DURDU: {durdu} → devam YOK. Hücre 6 ile zipla, şefe getir.')
elif not baslamis:
    print(f'{son} yazılmış ama hiç başlatılmamış → şimdi başlatılıyor')
    rc = kos(['python', 'orkestra/kuyruk.py', '--kok', '.', 'kos', son, '--kademeli', '--paralel', '1'])
    print(f'{son} rc={rc}')
else:
    kos(['python', 'orkestra/kosum.py', '--kok', '.', 'durum', '--yetim-damgala'])
    satirlar = [json.loads(l) for l in open(f'kuyruk/{AD}.jsonl', encoding='utf-8') if l.strip() and not l.startswith('#')]

    def basarili_seedler(varyant):
        s = set()
        for d in (KOK / 'runs').glob('*/'):
            c, dn = d / 'config.json', d / '.done'
            if c.exists() and dn.exists():
                cj, dj = json.loads(c.read_text(encoding='utf-8')), json.loads(dn.read_text(encoding='utf-8'))
                if cj.get('varyant') == varyant and dj.get('rc') == 0:
                    s.add(str(cj.get('seed')))
        return s

    a2_gecti = any('GEÇTİ' in p.read_text(encoding='utf-8').splitlines()[0]
                   for p in (KOK / 'sonuclar').glob(f'beklenti-{AD}*-asama2.md'))
    if a2_gecti:
        bitti = basarili_seedler('baseline')
        kalan = [j for j in satirlar if int(j.get('asama', 3)) == 3 and str(j['seed']) not in bitti]
        print('aşama 2 kapısı daha önce GEÇTİ; bitmiş aşama-3 seedleri:', sorted(bitti), '→ kalan', len(kalan))
    else:
        kalan = satirlar
        print('aşama 2 kapısı geçilmemiş/yok → bütün satırlar yeniden:', len(kalan))
    if not kalan:
        print('kalan koşum yok → hücre 6')
    else:
        YENI = f'{AD}-devam{len(devamlar) + 1}'
        (KOK / 'kuyruk' / f'{YENI}.jsonl').write_text(''.join(json.dumps(j, ensure_ascii=False) + '\n' for j in kalan), encoding='utf-8')
        shutil.copyfile(KOK / 'kuyruk' / f'{AD}.beklenti.json', KOK / 'kuyruk' / f'{YENI}.beklenti.json')
        t0 = time.time()
        rc = kos(['python', 'orkestra/kuyruk.py', '--kok', '.', 'kos', YENI, '--kademeli', '--paralel', '1'])
        print(f'\n{YENI} rc={rc} · süre {(time.time() - t0) / 60:.1f} dk  (0=başarılı, 1=başarısız koşum var, 3=kapıda DURDU)')

In [ ]:
# 6) Durum + aşama-3 kural raporu (bilgi; karar verici replikasyon kapısını şef yerelde koşar) + sonuç zip'i
kos(['python', 'orkestra/kosum.py', '--kok', '.', 'durum'])
kos(['python', 'orkestra/kuyruk.py', '--kok', '.', 'liste'])
for jl in sorted((KOK / 'kuyruk').glob('k3-baseline*.jsonl')):
    if (KOK / 'runs' / 'kuyruklar' / f'{jl.stem}.KUYRUK.done').exists():
        kos(['python', 'orkestra/beklenti_kontrol.py', '--kok', '.', '--kuyruk', jl.stem, '--asama', '3'])
ZIP = YZ / 'k3_baseline_sonuc.zip'
n = 0
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for klasor in ('runs', 'kuyruk', 'sonuclar'):
        for p in sorted((KOK / klasor).rglob('*')):
            if p.is_file() and p.suffix not in ('.pth', '.pt', '.tmp'):
                z.write(p, p.relative_to(KOK).as_posix()); n += 1
    for p in ('env/ortam-colab.json', 'env/requirements-colab.lock', 'data/indirme_colab.json', 'PAKET.json'):
        if (KOK / p).exists():
            z.write(KOK / p, p); n += 1
print(f'{ZIP} yazıldı · {n} dosya · {ZIP.stat().st_size / 1e6:.1f} MB')
print('İNDİR: Drive > Bilgisayarım > yz > k3_baseline_sonuc.zip → bilgisayarda WS/colab/gelen/ altına koy (açma, şef açar).')
try:
    from google.colab import files
    files.download(str(ZIP))
except Exception as e:
    print('otomatik indirme olmadı (sorun değil, Drive\'dan indir):', e)